In [1]:
import pandas as pd
import psycopg2

# -----------------------------------------
# 1. READ AND CLEAN THE DATA
# -----------------------------------------
df = pd.read_csv(r"C:\datasets\archive (3)\Amazon Sale Report.csv", low_memory=False)
print(f"Extraction Phase: Python sees exactly {len(df)} rows of raw data.")

# Snake casing
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ', '_')
df.columns = df.columns.str.replace('-', '_')

# Deleting unnecessary columns
df = df.drop(columns=['unnamed:_22', 'order_id', "index", "asin"])

# Convert date into datetime datatype
df["date"] = df["date"].astype("datetime64[ns]")

# Remove whitespace
whitespacecolunms = ['status', 'fulfilment', 'sales_channel_',
                     'ship_service_level', 'style', 'sku', 'category', 'size',
                     'courier_status', 'qty', 'currency', 'ship_country', 'b2b']

for col in whitespacecolunms:
    df[col] = df[col].astype(str).str.strip()

# Remove rows of amount where its blank
df = df.dropna(subset=['amount'])

# Filling blank values
df = df.fillna({
    'ship_city': "no_info",
    'ship_state': "no_info",
    'ship_postal_code': "no_info",
    'ship_country': "no_info",
    'promotion_ids': "not_available",
    'fulfilled_by': "not_available",
    'courier_status': "N.A"
})

# Created another row for index
df.insert(0, 'n', range(1, len(df) + 1))

# Force Qty to be a number (integer)
df['qty'] = pd.to_numeric(df['qty'], errors='coerce').fillna(0).astype(int)

# Force Postal Code to be a string
df['ship_postal_code'] = df['ship_postal_code'].astype(str)

# Fix the typo in sales_channel_ by renaming it
df = df.rename(columns={'sales_channel_': 'sales_channel'})


# -----------------------------------------
# 2. SAVE THE CLEAN DATA TO A CSV
# -----------------------------------------
print(f"Cleaning Phase Complete: Preparing to upload {len(df)} clean rows.")
df.to_csv('amazon_sales_clean.csv', index=False)


# -----------------------------------------
# 3. DEFINE THE TABLE BLUEPRINT
# -----------------------------------------
create_table_query = """
CREATE TABLE amazon_sales (
    n INT PRIMARY KEY,
    date DATE,
    status VARCHAR(100),
    fulfilment VARCHAR(50),
    sales_channel VARCHAR(50),
    ship_service_level VARCHAR(50),
    style VARCHAR(100),
    sku VARCHAR(250),
    category VARCHAR(100),
    size VARCHAR(50),
    courier_status VARCHAR(50),
    qty INT,
    currency VARCHAR(10),
    amount NUMERIC,
    ship_city VARCHAR(100),
    ship_state VARCHAR(100),
    ship_postal_code VARCHAR(50),
    ship_country VARCHAR(50),
    promotion_ids TEXT,
    b2b VARCHAR(10),
    fulfilled_by VARCHAR(50)
);
"""


# -----------------------------------------
# 4. CONNECT & UPLOAD TO POSTGRESQL
# -----------------------------------------
try:
    # Connect directly to PostgreSQL
    conn = psycopg2.connect(
        host="localhost",
        database="project",
        user="postgres",
        password="OnkPost1136",
        port="5432"
    )
    cursor = conn.cursor()
    print("✅ Connection succeeded to psql database!")

    # Build the strict Table Blueprint
    print("🛠️ Building the table structure...")
    cursor.execute("DROP TABLE IF EXISTS amazon_sales;")
    cursor.execute(create_table_query)
    conn.commit()
    print("✅ Table created successfully!")

    # Open the newly saved CSV and upload the data
    print("🚀 Uploading data...")
    with open('amazon_sales_clean.csv', 'r', encoding='utf-8') as f:
        sql_copy_query = "COPY amazon_sales FROM STDIN WITH CSV HEADER"
        cursor.copy_expert(sql_copy_query, f)

    # Save everything
    conn.commit()
    print("🎉 Upload complete! All rows inserted successfully.")

except Exception as e:
    print("❌ An error occurred:", e)

finally:
    if 'conn' in locals() and conn:
        cursor.close()
        conn.close()
        print("🔒 Database connection closed.")

Extraction Phase: Python sees exactly 128975 rows of raw data.
Cleaning Phase Complete: Preparing to upload 121180 clean rows.
✅ Connection succeeded to psql database!
🛠️ Building the table structure...
✅ Table created successfully!
🚀 Uploading data...
🎉 Upload complete! All rows inserted successfully.
🔒 Database connection closed.
